In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

In [ ]:
# 1. Scaled Dot-Product Attention
# q: [B, H, T, Dh], k/v: [B, H, S, Dh]
def scaled_dot_product_attention(q, k, v, mask=None):
    assert q.size(-1) == k.size(-1), "Q/K 的 head_dim 必须相同"
    assert k.size(-2) == v.size(-2), "K/V 的序列长度必须相同"
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))  # [B,H,T,S]
    if mask is not None:
        assert torch.broadcast_shapes(scores.shape, mask.shape) == scores.shape
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    attn = torch.softmax(scores, dim=-1)  # 沿 key 维 S 做 softmax
    return attn @ v, attn

B, H, T, S, Dh = 2, 4, 5, 6, 8
q = torch.randn(B, H, T, Dh, device=device)
k = torch.randn(B, H, S, Dh, device=device)
v = torch.randn(B, H, S, Dh, device=device)
out, attn = scaled_dot_product_attention(q, k, v)
assert out.shape == (B, H, T, Dh)
assert torch.allclose(attn.sum(-1), torch.ones(B, H, T, device=device), atol=1e-5)

In [ ]:
# 3. Cross Attention：Q 来自 decoder，K/V 来自 encoder
class CrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        B, L, D = x.shape
        return x.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, query, memory, memory_mask=None):
        B, T, D = query.shape
        assert memory.shape[0] == B and memory.shape[2] == D
        q = self.split_heads(self.wq(query))      # [B,H,T,Dh]
        k = self.split_heads(self.wk(memory))     # [B,H,S,Dh]
        v = self.split_heads(self.wv(memory))     # [B,H,S,Dh]
        mask = None if memory_mask is None else memory_mask[:, None, None, :]
        out, _ = scaled_dot_product_attention(q, k, v, mask)
        return self.wo(out.transpose(1, 2).contiguous().view(B, T, D))

cross_attn = CrossAttention(32, 4).to(device)
assert cross_attn(torch.randn(2, 3, 32, device=device), torch.randn(2, 7, 32, device=device)).shape == (2, 3, 32)